In [1]:
# ============================================================# Pengendali Motor ESP32 Berbasis Port Serial Serial# Fitur: Toggle Power & Kontrol Kecepatan (PWM)# ============================================================

import sys
import time
import serial
import serial.tools.list_ports

# Konfigurasi Utama
BAUD_RATE = 115200
CHIP_SIGNATURES = [
    "cp210", "cp2102", "cp210x",
    "ch340", "ch341", "ch34x",
    "ftdi", "ft232",
    "usb-serial", "usb serial",
    "usb uart", "uart bridge",
    "silicon labs",
    "espressif", "esp32"
]

HARDWARE_IDS = {
    (0x10C4, 0xEA60), (0x10C4, 0xEA70),
    (0x1A86, 0x7523), (0x1A86, 0x55D4),
    (0x0403, 0x6001), (0x0403, 0x6015),
    (0x303A, 0x1001)
}

INCREMENT_STEP = 10


def tampilkan_daftar_port():
    devices = serial.tools.list_ports.comports()
    if not devices:
        print("Peringatan: Tidak ditemukan port serial aktif.")
        return []
    
    print("Daftar Port Serial Tersedia:")
    for port_info in devices:
        id_info = f" [VID:PID={port_info.vid:04X}:{port_info.pid:04X}]" if port_info.vid and port_info.pid else ""
        print(f"  > {port_info.device} -> {port_info.description}{id_info}")
    return list(devices)


def cari_port_esp32(cetak_info=True):
    devices = serial.tools.list_ports.comports()
    if not devices:
        if cetak_info:
            print("[X] Port serial kosong.")
        return None

    kandidat_terbaik = []
    
    for port in devices:
        deskripsi = (port.description or "").lower()
        pembuat = (port.manufacturer or "").lower()
        info_hw = (port.hwid or "").lower()
        gabungan_teks = f"{deskripsi} {pembuat} {info_hw}"

        bobot = 0
        if port.vid is not None and port.pid is not None:
            if (port.vid, port.pid) in HARDWARE_IDS:
                bobot += 120
        
        for kata_kunci in CHIP_SIGNATURES:
            if kata_kunci in gabungan_teks:
                bobot += 45
                break
                
        if "bluetooth" in gabungan_teks:
            bobot -= 250

        if bobot > 0:
            kandidat_terbaik.append((bobot, port))

    if not kandidat_terbaik:
        tanpa_bt = [p for p in devices if "bluetooth" not in (p.description or "").lower()]
        if len(tanpa_bt) == 1:
            if cetak_info:
                print(f"[Info] Menggunakan port tunggal non-Bluetooth: {tanpa_bt[0].device}")
            return tanpa_bt[0].device
        if cetak_info:
            print("[X] Gagal mendeteksi chip ESP32 secara spesifik.")
        return None

    kandidat_terbaik.sort(key=lambda item: item[0], reverse=True)
    if cetak_info:
        print("Hasil Pemindaian Port ESP32:")
        for bobot, port in kandidat_terbaik:
            print(f"  [Skor: {bobot}] {port.device} ({port.description})")
            
    return kandidat_terbaik[0][1].device


def inisialisasi_koneksi(nama_port, baud=BAUD_RATE):
    try:
        koneksi = serial.Serial(nama_port, baud, timeout=1)
        time.sleep(2)
        print(f"Berhasil terhubung ke: {nama_port} pada {baud} baud.")
        return koneksi
    except serial.SerialException as err:
        print(f"Kesalahan koneksi pada {nama_port}: {err}")
        return None


def kirim_instruksi(koneksi_ser, pesan: str):
    koneksi_ser.write((pesan + "\n").encode("utf-8"))
    koneksi_ser.flush()
    time.sleep(0.12)

    while koneksi_ser.in_waiting > 0:
        respon = koneksi_ser.readline().decode("utf-8", errors="ignore").strip()
        if respon:
            print(f"[ESP32 Feedback] {respon}")
        time.sleep(0.04)


# Helper Fungsi Perintah Kontrol
def aktuator_hidup(ser):
    print(">> Mengirim Perintah: Motor ON")
    kirim_instruksi(ser, "1")

def aktuator_mati(ser):
    print(">> Mengirim Perintah: Motor OFF")
    kirim_instruksi(ser, "0")

def tambah_kecepatan(ser, step=INCREMENT_STEP):
    print(f">> Mengirim Perintah: Kecepatan Naik (+{step})")
    kirim_instruksi(ser, "+")

def kurangi_kecepatan(ser, step=INCREMENT_STEP):
    print(f">> Mengirim Perintah: Kecepatan Turun (-{step})")
    kirim_instruksi(ser, "-")

def atur_pwm_langsung(ser, nilai_pwm: int):
    nilai_pwm = max(0, min(255, int(nilai_pwm)))
    print(f">> Mengirim Perintah: Set PWM -> {nilai_pwm}")
    kirim_instruksi(ser, f"PWM:{nilai_pwm}")


if __name__ == "__main__":
    print("=" * 60)
    print("PROGRAM INTERFACE SERIAL ESP32")
    print("=" * 60)
    tampilkan_daftar_port()
    print("=" * 60)

    target_port = cari_port_esp32()
    if not target_port:
        print("Eksekusi dihentikan: Port target tidak ditemukan.")
        sys.exit(1)
        
    print(f"\nPort Terpilih untuk Komunikasi: {target_port}\n")
    kabel_serial = inisialisasi_koneksi(target_port, BAUD_RATE)

    if not kabel_serial:
        print("Inisialisasi port gagal.")
    else:
        try:
            print("\n--- Menjalankan Skenario Otomatis ---")
            aktuator_hidup(kabel_serial)
            time.sleep(1.5)

            for _ in range(3):
                tambah_kecepatan(kabel_serial)
                time.sleep(0.8)

            for _ in range(2):
                kurangi_kecepatan(kabel_serial)
                time.sleep(0.8)

            atur_pwm_langsung(kabel_serial, 180)
            time.sleep(2)

            aktuator_mati(kabel_serial)
            print("--- Skenario Otomatis Selesai ---\n")

            print("Panduan Menu Interaktif:")
            print("  [1]  -> Nyalakan Motor")
            print("  [0]  -> Matikan Motor")
            print("  [+]  -> Tambah Kecepatan")
            print("  [-]  -> Kurangi Kecepatan")
            print("  [pX] -> Atur PWM (contoh: p120)")
            print("  [q]  -> Keluar Program\n")

            while True:
                masukan_user = input("Masukkan Perintah: ").strip()

                if masukan_user == "q":
                    break
                elif masukan_user == "1":
                    aktuator_hidup(kabel_serial)
                elif masukan_user == "0":
                    aktuator_mati(kabel_serial)
                elif masukan_user == "+":
                    tambah_kecepatan(kabel_serial)
                elif masukan_user == "-":
                    kurangi_kecepatan(kabel_serial)
                elif masukan_user.lower().startswith("p") and masukan_user[1:].isdigit():
                    atur_pwm_langsung(kabel_serial, int(masukan_user[1:]))
                else:
                    print("Perintah tidak dikenali! Gunakan opsi yang tersedia.")

        except KeyboardInterrupt:
            print("\nProgram dihentikan secara manual oleh pengguna.")

        finally:
            try:
                aktuator_mati(kabel_serial)
            except Exception:
                pass
            kabel_serial.close()
            print("Koneksi serial berhasil ditutup.")

PROGRAM INTERFACE SERIAL ESP32
Daftar Port Serial Tersedia:
  > COM13 -> USB-Enhanced-SERIAL CH9102 (COM13) [VID:PID=1A86:55D4]
Hasil Pemindaian Port ESP32:
  [Skor: 120] COM13 (USB-Enhanced-SERIAL CH9102 (COM13))

Port Terpilih untuk Komunikasi: COM13

Berhasil terhubung ke: COM13 pada 115200 baud.

--- Menjalankan Skenario Otomatis ---
>> Mengirim Perintah: Motor ON
[ESP32 Feedback] LED is on !!!
>> Mengirim Perintah: Kecepatan Naik (+10)
>> Mengirim Perintah: Kecepatan Naik (+10)
>> Mengirim Perintah: Kecepatan Naik (+10)
>> Mengirim Perintah: Kecepatan Turun (-10)
>> Mengirim Perintah: Kecepatan Turun (-10)
>> Mengirim Perintah: Set PWM -> 180
[ESP32 Feedback] LED is on !!!
[ESP32 Feedback] LED is off !!!
>> Mengirim Perintah: Motor OFF
[ESP32 Feedback] LED is off !!!
--- Skenario Otomatis Selesai ---

Panduan Menu Interaktif:
  [1]  -> Nyalakan Motor
  [0]  -> Matikan Motor
  [+]  -> Tambah Kecepatan
  [-]  -> Kurangi Kecepatan
  [pX] -> Atur PWM (contoh: p120)
  [q]  -> Keluar Pr